Fetches and saves a set of CAPTCHA images from SCOTUS for use in training and evaluation.

In [ ]:
import json
from urllib.request import urlretrieve
from urllib.parse import urlparse, parse_qs
import random

import requests
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.wait import WebDriverWait

chrome_options = Options()
chrome_options.add_argument("--headless=new")
driver = webdriver.Chrome(options=chrome_options)
driver.implicitly_wait(1.0)

In [ ]:
case_ids = [
    "25-250",
    "25-261",
    "25a561"
]

In [ ]:
def get_captcha_id_source() -> dict[str, str]:
    # Pick a random case to fetch the CAPTCHA from so we don't hit any one case with abnormally high activity.
    driver.get('https://file.supremecourt.gov/casenotification?caseNumber=' + random.choice(case_ids))

    captcha_image = driver.find_element(by=By.CSS_SELECTOR, value='.k-captcha-image img')
    wait = WebDriverWait(driver, timeout=2)
    wait.until(lambda d: captcha_image.is_displayed())
    captcha_image_source = captcha_image.get_attribute('src')
    uuid = parse_qs(urlparse(captcha_image_source).query)['captchaId'][0]
    captcha_audio_source = "https://file.supremecourt.gov/Captcha/audio?captchaId=" + uuid
    print("Image source:", captcha_image_source)
    print("Audio source:", captcha_audio_source)

    return {
        "uuid": uuid,
        "image_source": captcha_image_source,
        "audio_source": captcha_audio_source
    }

In [ ]:
def download_captcha_image() -> None:
    id_source = get_captcha_id_source()

    image_save_path = "./captchas/image/" + id_source["uuid"] + ".png"
    urlretrieve(id_source["image_source"], image_save_path)
    print("Downloaded CAPTCHA image to:", image_save_path)

In [ ]:
for _ in range(1):
    download_captcha_image()

In [ ]:
def download_captcha_audio() -> None:
    id_source = get_captcha_id_source()
    uuid = id_source["uuid"]
    audio_source = id_source["audio_source"]

    audio_save_path = "./captchas/audio/" + uuid + ".wav"
    # For some reason, SCOTUS is annoying about the audio files, and I couldn't figure out a combination of headers that would let me request them directly without running into a 503. However, if we just run this, we can get the same effect and trick SCOTUS into thinking we're just a nice, normal user who needs the audio CAPTCHA.
    audio_bytes = bytearray(driver.execute_script("""
        return await fetch(\"""" + audio_source + """\").then((response) => response.bytes())
    """))

    with open(audio_save_path, "wb") as f:
        f.write(audio_bytes)

In [ ]:
for _ in range(100):
    download_captcha_audio()